<a href="https://colab.research.google.com/github/ywoo-create/osspteamproject/blob/ywoo/ossp_%EC%86%8C%EB%A6%AC%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B0%9B%EB%8A%94_%EC%95%8C%EA%B3%A0%EB%A6%AC%EC%A6%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf osspteamproject
!git clone -b dataset https://github.com/ywoo-create/osspteamproject.git
%cd osspteamproject

Cloning into 'osspteamproject'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 65 (delta 17), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 256.66 KiB | 6.58 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/osspteamproject


In [ ]:
!pip install librosa soundfile tensorflow scikit-learn matplotlib -q

In [ ]:
import os
import numpy as np
import librosa
import tensorflow as tf
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from IPython.display import HTML, display

SAMPLE_RATE = 22050
DURATION = 3.0
N_MELS = 64
MAX_LEN = 64

CLASS_DIRS = [
    "0_indoor_alarms",
    "1_outdoor_warnings",
    "2_emergency_alarms"
]

CLASS_LABELS = {
    0: "위험도 1: 실내 생활 알림음",
    1: "위험도 2: 실외 주의 소리",
    2: "위험도 3: 응급/위험 경보음"
}

ALERT_MESSAGES = {
    0: "실내 생활 알림음이 감지되었습니다. 주변을 확인하세요.",
    1: "실외 주의 소리가 감지되었습니다. 차량이나 주변 상황을 확인하세요.",
    2: "응급 또는 위험 경보음이 감지되었습니다. 즉시 주변 상황을 확인하세요."
}

ALERT_COLORS = {
    0: "#FFD966",
    1: "#F6B26B",
    2: "#E06666"
}

In [ ]:
AUDIO_EXTENSIONS = [".wav", ".m4a", ".mp3", ".aac", ".flac"]

def count_audio_files():
    for class_dir in CLASS_DIRS:
        folder = Path(class_dir)

        if not folder.exists():
            print(f"[오류] 폴더 없음: {class_dir}")
            continue

        files = [
            f for f in folder.rglob("*")
            if f.suffix.lower() in AUDIO_EXTENSIONS
        ]

        print(f"{class_dir}: {len(files)}개")

count_audio_files()

0_indoor_alarms: 1개
1_outdoor_warnings: 0개
2_emergency_alarms: 0개


In [ ]:
def fix_audio_length(y, sr):
    target_length = int(sr * DURATION)

    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)))
    else:
        y = y[:target_length]

    return y


def make_mel_spectrogram(y, sr):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    if mel_db.shape[1] < MAX_LEN:
        mel_db = np.pad(
            mel_db,
            ((0, 0), (0, MAX_LEN - mel_db.shape[1]))
        )
    else:
        mel_db = mel_db[:, :MAX_LEN]

    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

    return mel_db[..., np.newaxis]


def load_audio_file(file_path):
    y, sr = librosa.load(
        file_path,
        sr=SAMPLE_RATE,
        mono=True,
        duration=DURATION
    )

    y = fix_audio_length(y, sr)
    y = librosa.util.normalize(y)

    return y, sr

In [ ]:
def add_noise(y):
    noise = np.random.randn(len(y))
    return y + 0.005 * noise


def change_volume(y):
    rate = np.random.uniform(0.7, 1.3)
    return y * rate


def shift_time(y):
    shift = np.random.randint(-2000, 2000)
    return np.roll(y, shift)


def make_augmented_audio_list(y):
    augmented = []

    augmented.append(y)
    augmented.append(add_noise(y))
    augmented.append(change_volume(y))
    augmented.append(shift_time(y))

    return augmented

In [ ]:
def build_dataset_from_github_folders():
    X = []
    y_data = []

    for label_index, class_dir in enumerate(CLASS_DIRS):
        folder = Path(class_dir)

        if not folder.exists():
            print(f"[오류] 폴더 없음: {class_dir}")
            continue

        audio_files = [
            f for f in folder.rglob("*")
            if f.suffix.lower() in AUDIO_EXTENSIONS
        ]

        print(f"\n[{class_dir}] 원본 파일 수: {len(audio_files)}개")

        for file_path in audio_files:
            try:
                audio, sr = load_audio_file(str(file_path))

                augmented_audio_list = make_augmented_audio_list(audio)

                for aug_audio in augmented_audio_list:
                    mel = make_mel_spectrogram(aug_audio, sr)
                    X.append(mel)
                    y_data.append(label_index)

            except Exception as e:
                print(f"[처리 실패] {file_path} / 이유: {e}")

    X = np.array(X)
    y_data = np.array(y_data)

    print("\n데이터셋 생성 완료")
    print("X shape:", X.shape)
    print("y shape:", y_data.shape)

    return X, y_data


X, y_data = build_dataset_from_github_folders()


[0_indoor_alarms] 원본 파일 수: 1개


/tmp/ipykernel_1238/1976959942.py:35: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



[1_outdoor_warnings] 원본 파일 수: 0개

[2_emergency_alarms] 원본 파일 수: 0개

데이터셋 생성 완료
X shape: (4, 64, 64, 1)
y shape: (4,)


In [ ]:
unique, counts = np.unique(y_data, return_counts=True)

for label, count in zip(unique, counts):
    print(CLASS_LABELS[int(label)], ":", count, "개")

위험도 1: 실내 생활 알림음 : 4 개


In [ ]:
num_classes = len(CLASS_DIRS)

y_cat = to_categorical(y_data, num_classes=num_classes)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_data
)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64, 64, 1)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=8
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 683,011 (2.61 MB)

 Trainable params: 683,011 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6667 - loss: 1.0871 - val_accuracy: 1.0000 - val_loss: 0.8160
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 1.0000 - loss: 0.8118 - val_accuracy: 1.0000 - val_loss: 0.4481
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 1.0000 - loss: 0.4313 - val_accuracy: 1.0000 - val_loss: 0.1315
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 1.0000 - loss: 0.1469 - val_accuracy: 1.0000 - val_loss: 0.0151
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 1.0000 - loss: 0.0213 - val_accuracy: 1.0000 - val_loss: 7.7956e-04
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 1.0000 - loss: 0.0029 - val_accuracy: 1.0000 - val_loss: 2.2173e-05
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 1.0000 - loss: 1.1590e-04 - val_accuracy: 1.0000 - val_loss: 4.7684e-07
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step - accuracy: 1.0000 - loss: 2.5545e-04 - val_accuracy: 1

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print(f"테스트 정확도: {acc:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 1.0000 - loss: 0.0000e+00
테스트 정확도: 1.0000


In [ ]:
def create_alert_result(class_idx, confidence):
    if confidence < 0.7:
        return {
            "class_idx": class_idx,
            "label": "판단 보류",
            "confidence": confidence,
            "message": "소리가 명확하지 않습니다. 다시 확인해 주세요.",
            "color": "#B7B7B7"
        }

    return {
        "class_idx": class_idx,
        "label": CLASS_LABELS[class_idx],
        "confidence": confidence,
        "message": ALERT_MESSAGES[class_idx],
        "color": ALERT_COLORS[class_idx]
    }


def print_alert(alert_result):
    print("\n" + "=" * 50)
    print("[생활 소리 감지 결과]")
    print("분류:", alert_result["label"])
    print(f"신뢰도: {alert_result['confidence'] * 100:.2f}%")
    print("알림:", alert_result["message"])
    print("=" * 50)

In [ ]:
def predict_audio_file(file_path):
    audio, sr = load_audio_file(file_path)
    mel = make_mel_spectrogram(audio, sr)

    input_data = np.expand_dims(mel, axis=0)

    prediction = model.predict(input_data, verbose=0)[0]

    class_idx = int(np.argmax(prediction))
    confidence = float(prediction[class_idx])

    alert_result = create_alert_result(class_idx, confidence)
    print_alert(alert_result)

    return alert_result


test_files = [
    f for f in Path("0_indoor_alarms").rglob("*")
    if f.suffix.lower() in AUDIO_EXTENSIONS
]

print("테스트 가능 파일 수:", len(test_files))

if len(test_files) > 0:
    alert_result = predict_audio_file(str(test_files[0]))
else:
    print("테스트할 파일이 없습니다.")

테스트 가능 파일 수: 1


/tmp/ipykernel_1238/1976959942.py:35: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



[생활 소리 감지 결과]
분류: 위험도 1: 실내 생활 알림음
신뢰도: 100.00%
알림: 실내 생활 알림음이 감지되었습니다. 주변을 확인하세요.


In [ ]:
def show_alert_ui(alert_result):
    html = f"""
    <div style="
        width: 380px;
        padding: 30px;
        border-radius: 24px;
        background: {alert_result['color']};
        color: #111;
        font-family: Arial;
        text-align: center;
        box-shadow: 0 4px 16px rgba(0,0,0,0.2);
    ">
        <h2>소리 감지 결과</h2>
        <h1>{alert_result['label']}</h1>
        <p style="font-size: 20px;">{alert_result['message']}</p>
        <p>신뢰도: {alert_result['confidence'] * 100:.2f}%</p>
    </div>
    """

    display(HTML(html))


show_alert_ui(alert_result)